# Downloading Hansen Global Forest Change — `loss` band

Source: [UMD/hansen/global_forest_change_2024_v1_12](https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2024_v1_12). Native 30m; `loss` is binary (1 = forest loss during 2000–2024, 0 = no loss).

Export scale = **100m** to match the existing Hansen `treecover2000` download used in `forestloss_ethnologue.ipynb`. The `loss` band's default GEE pyramid policy is `mode`, so downsampling preserves the dominant class.

Exports go to Google Drive via Earth Engine. Monitor at the [GEE Task Manager](https://code.earthengine.google.com/tasks).

In [2]:
import ee
import geemap

ee.Authenticate()  # run once if not already authenticated
ee.Initialize()

In [3]:
# Global extent (same bbox used for the other Hansen export)
world_bbox = ee.Geometry.BBox(-180, -85, 180, 85)

treeloss = ee.Image("UMD/hansen/global_forest_change_2024_v1_12").select("loss")

print(treeloss.getInfo()["id"])  # sanity-check

UMD/hansen/global_forest_change_2024_v1_12


In [4]:
# Quick preview (red = loss)
vis_params = {"min": 0, "max": 1, "palette": ["ffffff", "ff0000"]}

Map = geemap.Map(center=[-7.5, -72.5], zoom=5)
Map.addLayer(treeloss.clip(world_bbox), vis_params, "Hansen loss 2000–2024")
Map.addLayer(world_bbox, {}, "Region")
Map

Map(center=[-7.5, -72.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [5]:
# Export to Google Drive at 100m (downsampled from native 30m)
task = ee.batch.Export.image.toDrive(
    image=treeloss,
    description="treeloss_100m_30m",
    folder="GEE_exports",
    fileNamePrefix="treeloss_100m_30m",
    region=world_bbox,
    scale=100,
    crs="EPSG:4326",
    maxPixels=1e13,
)

task.start()
print("Export task started:", task.id)

Export task started: BDFJNRU2AGN54UEVO7AXBLD3


### NOTE: track the export at the [GEE Task Manager](https://code.earthengine.google.com/tasks)

Drop the output tiles into `maps/raw/Hansen_forest/treeloss/` in the Dropbox project folder, matching the existing layout used by `forestloss_ethnologue.ipynb`.